# CHEN — Phase 1: Static Cascade

This notebook walks through **Phase 1** of CHEN: chaining small, specialized models in a fixed sequence, passing **text** between them.

**Goal of Phase 1:** Verify that chaining small models at all works, and establish a cost/quality baseline that Phase 2 (KV-cache passing) and Phase 3 (dynamic routing) build on.

We use the bundled `MockBackend` so the notebook runs in under 5 seconds on a CPU — no GPU, no model downloads. Swap `MockBackend` for `HuggingFaceBackend` to run with real models (see the README for the install command).

## 1. Setup

In [ ]:
from chen.backends.mock import MockBackend
from chen.core.expert import Expert, ExpertRole
from chen.phases.phase1_cascade import CascadePipeline

## 2. Build the expert garage

Three small models, each fine-tuned for a different latent task:

| Expert | Role | Size | Trained for |
|--------|------|------|-------------|
| `analyst` | ANALYST | 3B | Entity extraction, logic parsing |
| `reasoner` | REASONER | 8B | Math, code, multi-step reasoning |
| `synthesizer` | SYNTHESIZER | 3B | Latent → natural language |

In [ ]:
experts = [
    Expert(
        name="analyst",
        role=ExpertRole.ANALYST,
        backend=MockBackend(params_m=3_000, role_hint="analyst"),
        description="Extracts entities and logic from text.",
    ),
    Expert(
        name="reasoner",
        role=ExpertRole.REASONER,
        backend=MockBackend(params_m=8_000, role_hint="reasoner"),
        description="Math and code reasoning.",
    ),
    Expert(
        name="synthesizer",
        role=ExpertRole.SYNTHESIZER,
        backend=MockBackend(params_m=3_000, role_hint="synthesizer"),
        description="Converts latent state to natural language.",
    ),
]
print(f"Built {len(experts)} experts:")
for e in experts:
    print(f"  - {e.name:14s} {e.role.value:14s} {e.params_m // 1000}B params")

## 3. Build the cascade pipeline

Hard-code the sequence: `analyst → reasoner → synthesizer`. Each expert's text output is fed as the next expert's prompt. No KV-cache, no routing — just a text chain.

In [ ]:
pipe = CascadePipeline(
    experts=experts,
    sequence=["analyst", "reasoner", "synthesizer"],
    max_tokens_per_expert=128,
    write_intermediate_to_memory=True,
    memory_retrieve_k=2,
)
print(f"Pipeline sequence: {pipe.sequence}")

## 4. Run a prompt through the cascade

In [ ]:
prompt = (
    "Read this 10-page PDF about renewable energy trends and write a "
    "Python script that loads the data and plots solar vs wind capacity "
    "over time."
)
result = pipe.run(prompt)
print("=== Final output ===")
print(result.output)

## 5. Inspect per-expert metrics

In [ ]:
for i, m in enumerate(result.per_expert, 1):
    print(f"Expert {i}: {m.expert_name:14s} ({m.role.value})")
    print(f"  params:      {m.params_m}M")
    print(f"  in tokens:   {m.input_tokens}")
    print(f"  out tokens:  {m.output_tokens}")
    print(f"  latency:     {m.latency_ms:.2f} ms")
    print(f"  used KV:     {m.used_kv_cache}")
    print()

## 6. Aggregate metrics & cost

In [ ]:
print(result.metrics.summary())
print()
print(f"Total cost:      ${result.metrics.total_cost_usd:.6f}")
print(f"Cost / 1M tokens: ${result.metrics.cost_per_1m_tokens:.4f}")
print(f"EPU:              {result.metrics.epu:.3f}")

## 7. Inspect shared memory

Each expert wrote its output to the shared memory store. Later experts (or future prompts) can retrieve them.

In [ ]:
print(f"Memory entries: {pipe.memory.count()}")
for entry in pipe.memory._entries:
    print(f"  [{entry.role:14s}] {entry.text[:80]}...")

## 8. What did we just prove?

- A 3-expert cascade (3B + 8B + 3B = 14B params total) successfully processed a complex prompt end-to-end.
- The pipeline tracked per-expert latency, token counts, and cost.
- Each expert's output was written to shared memory — visible to later experts and to future prompts.
- The EPU (Effective Parameter Utilization) and cost-per-1M-tokens KPIs are computed.

## 9. What's next?

**Phase 2** replaces the text handoff with a **KV-cache handoff** — instead of converting each expert's output back to English, we pass the mathematical memory (KV-cache) directly. This is the key differentiator of CHEN. Open `02_phase2_kv_pass.ipynb` to see it.

**Phase 3** adds **dynamic routing** — a tiny classifier that decides which experts to wake up per prompt. Open `03_phase3_routing.ipynb` to see it.